In [7]:
import polars as pl
import re

DATA_PATH = "S:/campaignfinance/data/"

In [8]:
def remove_punc(text):
    pattern = r'[^a-zA-Z0]'
    cleaned_text = re.sub(pattern, '', text)
    return cleaned_text

def parse_last_name(x):
    suffixes = {"JR", "SR", "II", "III", "IV", "V"}

    before_comma = x.split(",")[0].strip()
    parts = before_comma.split()
    while parts and parts[-1].upper().strip(".") in suffixes:
        parts.pop()
    if not parts:
        return ""
    last = parts[-1].split("-")[-1]
    last_clean = ''.join(c for c in last if c.isalpha())

    return last_clean

In [9]:
contributions = pl.read_parquet(f"{DATA_PATH}FEC/processed/firms_principal_committees_contributions.parquet")
candidates = pl.read_csv(f"{DATA_PATH}FEC/API/candidates/candidate_history_H.csv",
                         schema_overrides={"address_zip": pl.Utf8})
elections = pl.read_csv(f"{DATA_PATH}electionratings/IE/house_ratings.csv")
elections = elections[:, 1:]
general_cands = pl.read_csv(f"{DATA_PATH}other/house_general_cands.csv")

In [10]:
# Process elections
elections = elections.with_columns([
    pl.col("date").str.to_datetime().alias("date"),
    pl.col("date").str.to_datetime().dt.year().alias("year")
])

# Aggregate ratings by taking the mean, and interpreting this as continuous values from -4 to 4
elections = (
    elections
    .group_by(["state", "district", "year"])
    .agg([
        pl.col("incumbent_party").first(),
        pl.col("incumbent").first(),
        pl.col("open").first(),
        pl.col("special").first(),
        pl.col("rating").mean()
    ])
)

elections = elections.with_columns([
    ((pl.col("incumbent_party") == "republican") & (pl.col("rating") < 0))
    .cast(pl.Int8)
    .alias("flip"),
    pl.col("rating").abs().alias("uncertainty")
])

In [11]:
# Process candidates
candidates = (
    candidates
    .drop_nulls(subset = ["candidate_election_year", "district"])
    .with_columns([
        pl.col("candidate_election_year").cast(pl.Int64).alias("year"),
        pl.col("district").cast(pl.Int64),
        pl.col("name").map_elements(parse_last_name, return_dtype=pl.Utf8).alias("last_name")
    ])
    .select([
        "candidate_id",
        "state", 
        "district",
        "year",
        "incumbent_challenge",
        "party",
        "last_name"
    ])
)

In [12]:
# Process contributions
firm_size = (
    contributions
    .filter(pl.col("amount") > 0)
    .group_by("cmte_id")
    .agg([
        pl.col("amount").sum().alias("firm_amount"),
        pl.col("cand_id").n_unique().alias("firm_candidates")
    ])
)
contributions= (
    contributions
    .filter(pl.col("cand_id").str.starts_with("H"))
    .select([
        "cmte_id", "name", "amount", "cand_id", "rcpt_cmte", "year", "month", "day",
        "cmte_nm", "connected_org", "corporation", "instrument", "cusip",
        "hq", "hq_state", "hq_city", "ric", "permid", "GICS_Sector", "GICS_Industry_Group",
        "GICS_Industry", "GIS_Subindustry", "TRBC_Econ_Sector", "TRBC_Business_Sector",
        "TRBC_Industry_Group", "TRBC_Industry", "TRBC_ID", "NAICS_Sector", "NAICS_Subsector",
        "NAICS_Industry_Group"
    ])
    .join(firm_size, on = "cmte_id", how = "left")
)

In [13]:
merged_data = (
    contributions
    .join(
        candidates,
        left_on = ["cand_id", "year"],
        right_on = ["candidate_id", "year"],
        how = "inner"
    )
    .with_columns(
        pl.when(pl.col("district") == 0)
        .then(1)
        .otherwise(pl.col("district"))
        .alias("district")
    )
    .join(
        elections,
        on = ["state", "district", "year"],
        how = "inner"
    )
    .filter(pl.col("amount") > 0)
)

In [14]:
# Race level data
products = (
    merged_data
    .group_by(["cmte_id", "year", "state", "district"])
    .agg([
        pl.col("amount").sum().alias("total_amount"),
        pl.col("amount").count().alias("total_count"),
        pl.col("party").n_unique().alias("nparty")
    ])
    .with_columns(pl.lit(1).alias("contribute"))
)

vars = ["cmte_id", "year", "corporation", "instrument", "cusip",
        "hq", "hq_state", "hq_city", "ric", "permid", "GICS_Sector", "GICS_Industry_Group",
        "GICS_Industry", "GIS_Subindustry", "TRBC_Econ_Sector", "TRBC_Business_Sector",
        "TRBC_Industry_Group", "TRBC_Industry", "TRBC_ID", "NAICS_Sector", "NAICS_Subsector",
        "NAICS_Industry_Group", "firm_amount", "firm_candidates"]

corporate_pacs = (
    contributions
    .select(vars)
    .unique()
)

corporate_pacs.write_parquet(f"{DATA_PATH}FEC/processed/firm_pacs.parquet")

race_data = (
    corporate_pacs
    .join(elections, on = "year", how = "inner")
    .join(
        products,
        on = ["cmte_id", "year", "state", "district"],
        how = "left"
    )
    .with_columns([
        pl.col("total_amount").fill_null(0),
        pl.col("total_count").fill_null(0),
        pl.col("nparty").fill_null(0),
        pl.col("contribute").fill_null(0)
    ])
)

race_data.write_parquet(f"{DATA_PATH}house_firm_race.parquet")

In [15]:
general_cands = (
    general_cands
    .filter(pl.col("year") >= 2010)
    .filter(~pl.col("writein"))
    .filter(~pl.col("party").is_null())
    .with_columns(
        pl.col("candidate").map_elements(parse_last_name, return_dtype=pl.Utf8).alias("last_name")
    )
)

vars = ["year", "state_po", "district", "candidate", "party", "candidatevotes", "totalvotes", "last_name"]
general_cands = (
    general_cands
    .select(vars)
    .rename({"state_po": "state"})
)

filter_cands = (
    general_cands.join(
        candidates,
        on = ["year", "state", "district", "last_name"],
        how = "left",
        suffix = "_y"
    )
    .drop_nulls(subset = ["candidate_id"])
    .rename({"party_y": "party_abb"})
    .select([
        "year", "state", "district", "candidate", "party",
        "candidatevotes", "totalvotes", "candidate_id", "incumbent_challenge"
    ])
    .with_columns(
        pl.when(pl.col("district") == 0)
        .then(1)
        .otherwise(pl.col("district"))
        .alias("district")
    )
    .join(elections, on = ["state", "district", "year"], how = "inner")
)

In [16]:
products = (
    merged_data
    .group_by(["cmte_id", "year", "cand_id"])
    .agg([
        pl.col("amount").sum().alias("total_amount"),
        pl.col("amount").count().alias("total_count"),
        pl.col("party").n_unique().alias("nparty")
    ])
    .rename({"cand_id": "candidate_id"})
    .with_columns(pl.lit(1).alias("contribute"))
)

cand_data = (
    corporate_pacs
    .join(filter_cands, on = "year", how = "inner")
    .join(products, on = ["cmte_id", "year", "candidate_id"], how = "left")
    .with_columns([
        pl.col("total_amount").fill_null(0),
        pl.col("total_count").fill_null(0),
        pl.col("nparty").fill_null(0),
        pl.col("contribute").fill_null(0)
    ])
)

cand_data.write_parquet(f"{DATA_PATH}house_firm_cand.parquet")

In [17]:
cand_data = pl.read_parquet(f"{DATA_PATH}house_firm_cand.parquet")

In [18]:
cand_data

cmte_id,year,corporation,instrument,cusip,hq,hq_state,hq_city,ric,permid,GICS_Sector,GICS_Industry_Group,GICS_Industry,GIS_Subindustry,TRBC_Econ_Sector,TRBC_Business_Sector,TRBC_Industry_Group,TRBC_Industry,TRBC_ID,NAICS_Sector,NAICS_Subsector,NAICS_Industry_Group,firm_amount,firm_candidates,state,district,candidate,party,candidatevotes,totalvotes,candidate_id,incumbent_challenge,incumbent_party,incumbent,open,special,rating,flip,uncertainty,total_amount,total_count,nparty,contribute
str,i64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32,str,i64,str,str,i64,i64,str,str,str,str,i64,i64,f64,i8,f64,f64,u32,u32,i32
"""C00126847""",2010,"""KAMAN""","""4295908621""","""483548103""","""United States of America""","""CONNECTICUT""","""Bloomfield""","""KAMN.N^D24""",4.2959e9,"""Industrials""","""Capital Goods""","""Aerospace & Defense""","""Aerospace & Defense""","""['Industrials']""","""['Industrial Goods']""","""['Aerospace & Defense']""","""['Aerospace & Defense']""","""['5210101010']""","""['Manufacturing', 'Manufacturi…","""['Fabricated Metal Product Man…","""['Other Fabricated Metal Produ…",142600.0,29,"""AL""",1,"""DAVID WALTER""","""CONSTITUTION""",26357,156281,"""H0AL01048""","""C""","""Republican""","""Bonner""",0,0,-4.0,0,4.0,0.0,0,0,0
"""C00126847""",2010,"""KAMAN""","""4295908621""","""483548103""","""United States of America""","""CONNECTICUT""","""Bloomfield""","""KAMN.N^D24""",4.2959e9,"""Industrials""","""Capital Goods""","""Aerospace & Defense""","""Aerospace & Defense""","""['Industrials']""","""['Industrial Goods']""","""['Aerospace & Defense']""","""['Aerospace & Defense']""","""['5210101010']""","""['Manufacturing', 'Manufacturi…","""['Fabricated Metal Product Man…","""['Other Fabricated Metal Produ…",142600.0,29,"""AL""",1,"""JO BONNER""","""REPUBLICAN""",129063,156281,"""H2AL01077""","""I""","""Republican""","""Bonner""",0,0,-4.0,0,4.0,0.0,0,0,0
"""C00126847""",2010,"""KAMAN""","""4295908621""","""483548103""","""United States of America""","""CONNECTICUT""","""Bloomfield""","""KAMN.N^D24""",4.2959e9,"""Industrials""","""Capital Goods""","""Aerospace & Defense""","""Aerospace & Defense""","""['Industrials']""","""['Industrial Goods']""","""['Aerospace & Defense']""","""['Aerospace & Defense']""","""['5210101010']""","""['Manufacturing', 'Manufacturi…","""['Fabricated Metal Product Man…","""['Other Fabricated Metal Produ…",142600.0,29,"""AL""",2,"""BOBBY BRIGHT""","""DEMOCRAT""",106865,219028,"""H8AL02130""","""I""","""Democrat""","""Bright""",0,0,-0.181818,0,0.181818,0.0,0,0,0
"""C00126847""",2010,"""KAMAN""","""4295908621""","""483548103""","""United States of America""","""CONNECTICUT""","""Bloomfield""","""KAMN.N^D24""",4.2959e9,"""Industrials""","""Capital Goods""","""Aerospace & Defense""","""Aerospace & Defense""","""['Industrials']""","""['Industrial Goods']""","""['Aerospace & Defense']""","""['Aerospace & Defense']""","""['5210101010']""","""['Manufacturing', 'Manufacturi…","""['Fabricated Metal Product Man…","""['Other Fabricated Metal Produ…",142600.0,29,"""AL""",2,"""MARTHA ROBY""","""REPUBLICAN""",111645,219028,"""H0AL02087""","""C""","""Democrat""","""Bright""",0,0,-0.181818,0,0.181818,0.0,0,0,0
"""C00126847""",2010,"""KAMAN""","""4295908621""","""483548103""","""United States of America""","""CONNECTICUT""","""Bloomfield""","""KAMN.N^D24""",4.2959e9,"""Industrials""","""Capital Goods""","""Aerospace & Defense""","""Aerospace & Defense""","""['Industrials']""","""['Industrial Goods']""","""['Aerospace & Defense']""","""['Aerospace & Defense']""","""['5210101010']""","""['Manufacturing', 'Manufacturi…","""['Fabricated Metal Product Man…","""['Other Fabricated Metal Produ…",142600.0,29,"""AL""",3,"""MIKE ROGERS""","""REPUBLICAN""",117736,198139,"""H2AL03032""","""I""","""Republican""","""Rogers""",0,0,-4.0,0,4.0,0.0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""C00144576""",2010,"""BAKER MANOCK JENSEN A PROFESSI